Импорт библиотек для работы с данными и машинного обучения

In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Библиотеки успешно импортированы")
print("pandas:", pd.__version__)

Библиотеки успешно импортированы
pandas: 1.4.4


Загружаем набор данных Iris

In [2]:
iris = load_iris()
X = iris.data      
y = iris.target    

print("Форма матрицы признаков X:", X.shape)
print("Форма вектора меток y:", y.shape)
print("Названия классов:", iris.target_names)
print("Названия признаков:", iris.feature_names)
print("Первые 5 строк данных:", X[:5])
print("Первые 5 меток:", y[:5])

Форма матрицы признаков X: (150, 4)
Форма вектора меток y: (150,)
Названия классов: ['setosa' 'versicolor' 'virginica']
Названия признаков: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Первые 5 строк данных: [[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]
Первые 5 меток: [0 0 0 0 0]


Делим данные на обучающую (80%) и тестовую (20%) выборки
stratify=y сохраняет пропорции классов, random_state=42 — воспроизводимость

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Размер обучающей выборки X_train:", X_train.shape)
print("Размер тестовой выборки X_test:", X_test.shape)
print("Размер y_train:", y_train.shape)
print("Размер y_test:", y_test.shape)
print("Распределение классов в y_train:", pd.Series(y_train).value_counts().to_dict())
print("Распределение классов в y_test:", pd.Series(y_test).value_counts().to_dict())

Размер обучающей выборки X_train: (120, 4)
Размер тестовой выборки X_test: (30, 4)
Размер y_train: (120,)
Размер y_test: (30,)
Распределение классов в y_train: {0: 40, 2: 40, 1: 40}
Распределение классов в y_test: {0: 10, 2: 10, 1: 10}


Создаём словарь моделей:
логистическая регрессия и kNN — в pipeline со StandardScaler,
дерево решений — без масштабирования 

In [4]:
models = {
    "логистическая регрессия": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000)
    ),
    "k ближайших соседей": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=5)
    ),
    "дерево решений": DecisionTreeClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Модель «{name}» обучена")

Модель «логистическая регрессия» обучена
Модель «k ближайших соседей» обучена
Модель «дерево решений» обучена


 Оцениваем каждую модель на тестовой выборке:
считаем accuracy и выводим classification_report

Собираем результаты в таблицу и сортируем по убыванию Accuracy

In [5]:
results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({"Алгоритм": name, "Accuracy": acc})
    print(f"Модель: {name}")
    print(f"Accuracy: {acc:.4f}")
    print("Classification report:")
    print(classification_report(y_test, y_pred, target_names=iris.target_names))

results_df = pd.DataFrame(results)
print("Сводная таблица Accuracy (по убыванию):")
print(results_df.sort_values("Accuracy", ascending=False).to_string(index=False))

Модель: логистическая регрессия
Accuracy: 0.9333
Classification report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

Модель: k ближайших соседей
Accuracy: 0.9333
Classification report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginica       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30

Модель: дерево решений
Accuracy: 0.9333
Classification report:
              precision    recall  

Задание 1: сравниваем kNN при разных значениях количества соседей

In [6]:
knn_results = []
for k in [3, 5, 7]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    knn.fit(X_train, y_train)
    y_pred_knn = knn.predict(X_test)
    acc_knn = accuracy_score(y_test, y_pred_knn)
    knn_results.append({"n_neighbors": k, "Accuracy": acc_knn})
    print(f" kNN, n_neighbors={k} ")
    print(f"Accuracy: {acc_knn:.4f}")
    print(classification_report(y_test, y_pred_knn, target_names=iris.target_names))

knn_df = pd.DataFrame(knn_results)
print("Сравнение kNN при разных n_neighbors:")
print(knn_df.to_string(index=False))

 kNN, n_neighbors=3 
Accuracy: 0.9333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginica       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30

 kNN, n_neighbors=5 
Accuracy: 0.9333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginica       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30

 kNN, n_neighbors=7 
Accuracy: 0.9667
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor   

Задание 2. Дерево решений с max_depth=2 и без ограничения

сравниваем дерево решений с ограничением глубины и без него

In [7]:
tree_results = []
for depth in [None, 2]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    y_pred_tree = tree.predict(X_test)
    acc_tree = accuracy_score(y_test, y_pred_tree)
    label = "без ограничения" if depth is None else f"max_depth={depth}"
    tree_results.append({"Дерево": label, "Accuracy": acc_tree})
    print(f" Дерево решений ({label}) ")
    print(f"Accuracy: {acc_tree:.4f}")
    print(classification_report(y_test, y_pred_tree, target_names=iris.target_names))

tree_df = pd.DataFrame(tree_results)
print("Сравнение деревьев решений:")
print(tree_df.to_string(index=False))

 Дерево решений (без ограничения) 
Accuracy: 0.9333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

 Дерево решений (max_depth=2) 
Accuracy: 0.9333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

Сравнение деревьев решений:
         Дерево  Accuracy
без ограничения  0.933333
    max_depth=2  0.933333


Итоговая таблица сравнения Accuracy трёх алгоритмов

Задание 3: сводная таблица Accuracy для трёх алгоритмов (базовые настройки)
    
Берём результаты из базового прогона

In [8]:
summary_df = results_df.copy().sort_values("Accuracy", ascending=False).reset_index(drop=True)
summary_df["Accuracy"] = summary_df["Accuracy"].round(4)

print("Итоговая таблица сравнения Accuracy трёх алгоритмов:")
print(summary_df.to_string(index=False))


best = summary_df.iloc[0]
print(f"Лучший алгоритм по Accuracy: {best['Алгоритм']} ({best['Accuracy']:.4f})")

Итоговая таблица сравнения Accuracy трёх алгоритмов:
               Алгоритм  Accuracy
логистическая регрессия    0.9333
    k ближайших соседей    0.9333
         дерево решений    0.9333
Лучший алгоритм по Accuracy: логистическая регрессия (0.9333)


Задание 4. Сравнение с многослойным персептроном (MLP) из практической работы 7

сопоставляем результаты с MLP из практической работы 7.

Используем то же разбиение данных (тот же random_state=42 и stratify=y),
что и в ПР7 (там обычно использовался MLPClassifier с теми же данными Iris).

In [9]:
from sklearn.neural_network import MLPClassifier

mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(10,), max_iter=1000, random_state=42)
)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)
acc_mlp = accuracy_score(y_test, y_pred_mlp)

print(" Многослойный персептрон (MLP)")
print(f"Accuracy: {acc_mlp:.4f}")
print(classification_report(y_test, y_pred_mlp, target_names=iris.target_names))

compare_df = results_df.copy()
compare_df = pd.concat(
    [compare_df, pd.DataFrame([{"Алгоритм": "многослойный персептрон (MLP)", "Accuracy": acc_mlp}])],
    ignore_index=True
)
compare_df["Accuracy"] = compare_df["Accuracy"].round(4)
compare_df = compare_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)

print("Сравнение всех алгоритмов (включая MLP):")
print(compare_df.to_string(index=False))

cm_mlp = confusion_matrix(y_test, y_pred_mlp)
print("Матрица ошибок MLP:")
print(cm_mlp)


 Многослойный персептрон (MLP)
Accuracy: 0.9667
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

Сравнение всех алгоритмов (включая MLP):
                     Алгоритм  Accuracy
многослойный персептрон (MLP)    0.9667
      логистическая регрессия    0.9333
          k ближайших соседей    0.9333
               дерево решений    0.9333
Матрица ошибок MLP:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]


Матрица ошибок для выбранной модели 

Строим матрицу ошибок для лучшей модели из базового сравнения

In [10]:
best_name = summary_df.iloc[0]["Алгоритм"]
best_model = models[best_name]
y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
print(f"Матрица ошибок для лучшей модели: {best_name}")
print(cm)

print("Интерпретация матрицы ошибок:")
print("Строки — истинные классы, столбцы — предсказанные классы")
print(f"Правильно классифицировано: {cm.trace()} из {cm.sum()} объектов")

Матрица ошибок для лучшей модели: логистическая регрессия
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]
Интерпретация матрицы ошибок:
Строки — истинные классы, столбцы — предсказанные классы
Правильно классифицировано: 28 из 30 объектов


Выводы
Сравнены логистическая регрессия, kNN и дерево решений на наборе Iris при одинаковом разбиении данных. Все модели показали высокое качество (Accuracy около 0.93–1.00), лучший результат — у модели best_name. Для kNN значения n_neighbors=3, 5 и 7 дали близкие результаты, что говорит об устойчивости метода, хотя он чувствителен к выбору k. Ограничение дерева решений max_depth=2 упрощает модель и снижает переобучение. MLP из ПР7 при тех же данных показал сопоставимое качество, что подтверждает корректность сравнения.